### FES Acatlán, UNAM
## Módulo 3. Diplomado introducción a la Ciencia de Datos
### Oscar Gutiérrez Leal


# ENSAFI 2023 – Extracción de datos (documentación)

Este notebook documenta el proceso de conexión a la base de datos y
las consultas SQL utilizadas para generar los archivos CSV empleados
en los análisis y visualizaciones.

Este notebook es **documental**.  
El código no está diseñado para ejecutarse directamente.


## 1. Conexión a la base de datos

Se utilizó PostgreSQL para integrar los microdatos de la ENSAFI 2023.
La conexión se realizó desde Jupyter utilizando SQLAlchemy.


In [1]:
# # importar librerias para manejo de sql, dataframe y gráficos
# from sqlalchemy import create_engine
# from sqlalchemy import text
# import pandas as pd

# # Cargar las credenciales desde el archivo py con datos de base de datos local (no incluido)
# import configPost1

# # Extraer las credenciales
# host = configPost1.host
# database = configPost1.database
# user = configPost1.user
# password = configPost1.password
# port=configPost1.port

# # Crear la conexión
# engine = create_engine(f'postgresql://{user}:{password}@{host}:{port}/{database}')

# # definimos una consulta que soporte parametros y convierta la QUERY en cadena de forma adecuada
# def consulta(QUERY,engine, params=None):
#     return pd.read_sql_query(text(QUERY),engine,params=params)

# # Definir consulta en SQL compatible con postgresql
# try:
#     QUERY = """
#     SELECT tablename 
#     FROM pg_catalog.pg_tables
#     WHERE schemaname NOT IN ('pg_catalog','information_schema');
#     """
# # Ejecutar la consulta y obtener los resultados en un DataFrame
#     df = consulta(QUERY, engine)
# # Mostrar los primeros resultados
#     print("La conexión se ha realizado con éxito.\n")
#     print(df)
# except Exception as e:
#     print("Hubo un error al realizar la conexión:", e)

La conexión se ha realizado con éxito.

           tablename
0            tmodulo
1            entidad
2          municipio
3          tvivienda
4             thogar
5              tsdem
6  cat_rango_ingreso
7         cat_region
8           cat_tloc


## 2. Consulta1: Proporción de hogares con y sin privación



In [2]:
# Agrupacion usando subquerys agrupando por rango de ingresos. Se cargan etiquetas de los catálogos para claridad
# QUERY = """
# SELECT 
# 	e.rango_id,
# 	r.rango_descripcion,
# 	SUM(e.flag_privaciones) AS hog_privaciones,
# 	COUNT(*) AS tot_hogares,
# 	SUM(e.flag_privaciones)::FLOAT / COUNT(*) AS prop_privacion,
# 	1-(SUM(e.flag_privaciones)::FLOAT / COUNT(*)) AS prop_no_privacion
# FROM (SELECT 
# 	    llavehog,
# 	    p4_3 AS rango_id,
# 	    CASE WHEN										--1 significa que no hubo privacion
# 		(												--2 significa que hubo una privacion para la categoria
# 		CASE WHEN p4_10_1 = '2' THEN 1 ELSE 0 END +  	--9 significa que no sabe 
# 	    CASE WHEN p4_10_2 = '2' THEN 1 ELSE 0 END +
# 	    CASE WHEN p4_10_3 = '2' THEN 1 ELSE 0 END +
# 	    CASE WHEN p4_10_4 = '2' THEN 1 ELSE 0 END +
# 	    CASE WHEN p4_10_5 = '2' THEN 1 ELSE 0 END +      --1 significa que se identifica al menos 1 privacion
# 	    CASE WHEN p4_10_6 = '2' THEN 1 ELSE 0 END
#         ) >= 1 THEN 1 ELSE 0 END AS flag_privaciones   --0 significa que no se identifica alguna privacion (no que no haya)    
# 	FROM thogar											
# 	WHERE NOT (
# 	    p4_10_1 = '9' AND
# 	    p4_10_2 = '9' AND
# 	    p4_10_3 = '9' AND
# 	    p4_10_4 = '9' AND
# 	    p4_10_5 = '9' AND
#         p4_10_6 = '9'
# 	)) AS e
# INNER JOIN cat_rango_ingreso r ON r.rango_id = e.rango_id
# GROUP BY  e.rango_id,r.rango_descripcion
# ORDER BY e.rango_id;
# """
# pd.options.display.html.use_mathjax = False
# consulta1 = consulta(QUERY, engine)
# consulta1

,rango_id,rango_descripcion,hog_privaciones,tot_hogares,prop_privacion,prop_no_privacion
0,01,"Hasta $3,600",3377,4020,0.840050,0.159950
1,02,"De $3,601 a $6,100",3541,4440,0.797523,0.202477
2,03,"De $6,101 a $8,100",1949,2639,0.738537,0.261463
3,04,"De $8,101 a $9,900",847,1238,0.684168,0.315832
4,05,"De $9,901 a $12,100",1491,2432,0.613076,0.386924
5,06,"De $12,101 a $14,500",582,1008,0.577381,0.422619
6,07,"De $14,501 a $17,600",632,1268,0.498423,0.501577
7,08,"De $17,601 a $21,900",463,1201,0.385512,0.614488
8,09,"De $21,901 a $29,100",291,903,0.322259,0.677741
9,10,"De $29,101 a $45,100",192,857,0.224037,0.775963


## 3. Consulta2: Promedio de estrés emocional


In [3]:
# #Agrupación usando subquerys en formato largo para mantener el atributo de privacion desagregado
# QUERY = """
# SELECT
# 	pe.rango_id,
# 	pe.rango_descripcion,
# 	pe.flag_privacion,
# 	AVG(nivel_emociones_acum) AS promedio_emociones_acum
# FROM
# 	(SELECT
# 		e.llavehog,
# 		e.rango_id,
# 		r.rango_descripcion,
# 		e.flag_privacion,
# 		m.nivel_emociones_acum
# 	FROM
# 	(SELECT
# 		llavehog,
# 		p4_3 AS rango_id,
# 		CASE WHEN(
# 			CASE WHEN p4_10_1 = '2' THEN 1 ELSE 0 END+
# 			CASE WHEN p4_10_2 = '2' THEN 1 ELSE 0 END+
# 			CASE WHEN p4_10_3 = '2' THEN 1 ELSE 0 END+
# 			CASE WHEN p4_10_4 = '2' THEN 1 ELSE 0 END+
# 			CASE WHEN p4_10_5 = '2' THEN 1 ELSE 0 END+
#             CASE WHEN p4_10_6 = '2' THEN 1 ELSE 0 END
# 		 )>=1 THEN 'con privacion' 
# 		 ELSE 'sin privacion' END AS flag_privacion
# 	FROM thogar
# 	WHERE NOT (
# 		p4_10_1 = '9' AND
# 		p4_10_2 = '9' AND
# 		p4_10_3 = '9' AND
# 		p4_10_4 = '9' AND
# 		p4_10_5 = '9' AND
#         p4_10_6 = '9'
# 	)) e
# 	INNER JOIN (
# 		SELECT
# 			llavehog,
# 			(CASE WHEN p8_2_1 = '1' THEN 1 ELSE 0 END+
# 			 CASE WHEN p8_2_2 = '1' THEN 1 ELSE 0 END+
# 			 CASE WHEN p8_2_3 = '1' THEN 1 ELSE 0 END+
# 			 CASE WHEN p8_2_4 = '1' THEN 1 ELSE 0 END)
# 			 AS nivel_emociones_acum
# 		FROM tmodulo) m
# 		ON m.llavehog = e.llavehog
# 	INNER JOIN cat_rango_ingreso  r ON r.rango_id = e.rango_id) pe
# GROUP BY pe.rango_id, pe.rango_descripcion, pe.flag_privacion
# ORDER BY 3;
# """
# pd.options.display.html.use_mathjax = False
# consulta2 = consulta(QUERY, engine)
# consulta2

,rango_id,rango_descripcion,flag_privacion,promedio_emociones_acum
0,01,"Hasta $3,600",con privacion,1.815517
1,02,"De $3,601 a $6,100",con privacion,1.794973
2,03,"De $6,101 a $8,100",con privacion,1.741406
3,04,"De $8,101 a $9,900",con privacion,1.691854
4,05,"De $9,901 a $12,100",con privacion,1.666667
5,06,"De $12,101 a $14,500",con privacion,1.757732
6,07,"De $14,501 a $17,600",con privacion,1.653481
7,08,"De $17,601 a $21,900",con privacion,1.598272
8,09,"De $21,901 a $29,100",con privacion,1.639175
9,10,"De $29,101 a $45,100",con privacion,1.531250


## 4. Exportación de resultados

Los resultados de ambas consultas se exportaron como archivos CSV,
los cuales son utilizados posteriormente en el notebook de visualización.


In [4]:
# #Exportar los dataframe a csv para visualización
# consulta1.to_csv('consulta1_privaciones.csv', index = False)
# consulta2.to_csv('consulta2_estres.csv', index = False)

#### Autor: Oscar Gutiérrez Leal
#### Diplomado: Introducción a la Ciencia de Datos
#### Fuente de Datos: Encuesta Nacional sobre Salud Financiera (ENSAFI) 2023. INEGI
#### Fecha de última actualización: Enero 2026
#### Repositorio: https://github.com/ockis2403/ensafi2023
